In [1]:
import torch
import torch.nn as nn
import torch.utils.data
import torch.nn.functional as F
import numpy as np
import torchtext.vocab
from tqdm.notebook import tqdm
import time
import random
import matplotlib.pyplot as plt
import collections
import gc

d:\Miniconda\envs\pytorch_env\Lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
d:\Miniconda\envs\pytorch_env\Lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


In [2]:
seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True


In [3]:
class DebugTimer:
    def __init__(self, name=None):
        self.start_time = None
        self.name = name

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            self.timer_start(self.name)
            result = func(*args, **kwargs)
            self.timer_stop()
            return result

        return wrapper

    def timer_start(self, name=None):
        self.start_time = time.perf_counter()
        if name is not None:
            self.name = name
        print(f"{self.name}:", end="")

    def timer_stop(self):
        elapsed_time = round(time.perf_counter() - self.start_time, 4)
        print(f"{elapsed_time}s")


t = DebugTimer()

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class CausalConv1d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        dilation=1,
        groups=1,
        bias=True,
    ):
        super(CausalConv1d, self).__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=self.pad,
            dilation=dilation,
            groups=groups,
            bias=bias,
        )

    def forward(self, input):
        return self.conv(input)[:, :, : -self.pad]

@dataclass
class TestModelArgs:
    d_model: int
    d_inner: int
    n_layers: int
    vocab_size: int
    seq_max_len: int
    d_conv: int = 3
    vocab_size: int
    conv_bias: bool = True
    bias: bool = False
    dropout: float = 0.1


class TestModelBlock(nn.Module):
    '''FakeMambaBlock'''
    def __init__(self, args: TestModelArgs):
        super().__init__()
        self.d_inner = args.d_inner
        self.in_proj = nn.Linear(args.d_model, args.d_inner * 2, bias=args.bias)
        self.out_proj = nn.Linear(args.d_inner, args.d_model, bias=args.bias)
        self.down_conv = CausalConv1d(
            in_channels=args.d_inner,
            out_channels=args.d_model,
            bias=args.conv_bias,
            kernel_size=args.d_conv,
            groups=args.d_model,
        )
        # self.down_conv2 = CausalConv1d(
        #     in_channels=args.d_model,
        #     out_channels=args.d_inner,
        #     bias=args.conv_bias,
        #     kernel_size=args.d_conv,
        #     groups=args.d_model,
        # )
        self.up_conv = CausalConv1d(
            in_channels=args.d_model,
            out_channels=args.d_inner,
            bias=args.conv_bias,
            kernel_size=args.d_conv,
            groups=args.d_model,
        )
        self.rnn = nn.LSTM(
            args.d_model, args.d_model, dropout=args.dropout, batch_first=True
        )
        self.norm = RMSNorm(args.d_model)

    def forward(self, x):

        x_and_res = self.in_proj(x)  # shape (b, l, 2 * d_in)
        (x, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)
        x = self.down_conv(x.transpose(1, 2)).transpose(1, 2)
        x = F.silu(x)
        y, _ = self.rnn(x)
        y = self.up_conv(y.transpose(1, 2)).transpose(1, 2)
        y = F.silu(y)
        y = y * F.silu(res)
        output = self.out_proj(y)
        return output



class TestModel(nn.Module):
    '''FakeMambaSimple'''
    def __init__(self, args: TestModelArgs):
        super().__init__()
        self.model_args = args
        self.emb = nn.Embedding(args.vocab_size, args.d_model)
        self.blocks = nn.ModuleList([
                nn.ModuleList([TestModelBlock(args), RMSNorm(args.d_model)])
                for _ in range(args.n_layers)
            ])
        self.norm_f = RMSNorm(args.d_model)
        self.lm_head = nn.Linear(args.d_model, args.vocab_size, bias=False)
        # self.lm_head.weight = self.emb.weight

    def forward(self, x):
        x = self.emb(x)
        # for layer in self.layers:
        #     x = layer(x)
        for mixer, norm in self.blocks:
            res = x
            x = mixer(norm(x))
            # x = x + res
        x = self.norm_f(x)
        x = self.lm_head(x)
        return x
    

In [5]:
class TextDatasetV3(torch.utils.data.Dataset):
    def __init__(self, data_dir, vocab_size, downsample):
        super().__init__()
        self.tokenizer = lambda x: x.split(" ")
        self.load_and_preprocess_data(data_dir, downsample)
        self.build_vocab(vocab_size)
        self.encode_data()
        self.seq_max_len = len(self.input_data[0])  # padding后元素一样长

    @DebugTimer("加载并生成训练数据")
    def load_and_preprocess_data(self, data_dir, downsample):
        with open(data_dir, encoding="utf-8") as f:
            lines = f.read().splitlines()[::downsample]
        self.input_data = [self.tokenizer(line)[:-1] for line in lines]
        self.output_data = [self.tokenizer(line)[1:] for line in lines]

    @DebugTimer("构建词典")
    def build_vocab(self, vocab_size):
        all_words = [
            word for line in self.input_data + self.output_data for word in line
        ]
        word_counts = collections.Counter(all_words).most_common(vocab_size)
        self.vocab = Vocab(word_counts, specials=["<PAD>", "<UNK>"])
        self.word2index = self.vocab.stoi
        self.index2word = self.vocab.itos
        self.word_nums = len(self.vocab)

    @DebugTimer("编码数据")
    def encode_data(self):
        word2index = self.word2index
        self.input_data = [
            torch.tensor(
                [word2index.get(word, self.vocab.default_index) for word in sentence]
            )
            for sentence in tqdm(self.input_data)]
        self.output_data = [
            torch.tensor(
                [word2index.get(word, self.vocab.default_index) for word in sentence]
            )
            for sentence in tqdm(self.output_data)]
        
        self.input_data = torch.nn.utils.rnn.pad_sequence(
            self.input_data, batch_first=True
        )
        self.output_data = torch.nn.utils.rnn.pad_sequence(
            self.output_data, batch_first=True
        )

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        """
        根据索引获取数据样本
        参数:
            index (int): 索引值
        返回值:
            tuple: 输入数据和输出数据的元组，以tensor形式表示
        """
        return (
            self.input_data[index].clone().detach(),
            self.output_data[index].clone().detach().long(),
        )


class Vocab:
    def __init__(self, word_counts, specials=["<PAD>", "<UNK>"]):

        # 先添加特殊token到词典
        self.stoi = {}
        self.itos = []
        for special in specials:
            self.stoi[special] = len(self.stoi)

        # 添加普通单词
        for word, _ in word_counts:
            self.stoi.setdefault(word, len(self.stoi))

        # 设置默认索引为`<UNK>`的索引
        self.itos = list(self.stoi.keys())
        self.default_index = self.stoi["<UNK>"]

    def __call__(self, word):
        """
        调用这个方法比直接调用
        self.stoi.get(word, self.default_index)
        慢很多...
        """
        # 如果单词不在词典中，返回`<UNK>`的索引
        return self.stoi.get(word, self.default_index)

    def __len__(self):
        return len(self.stoi)

In [6]:
class TextGenerator:
    def __init__(self, model: nn.Module, vocab, device) -> None:
        self.model = model
        self.vocab = vocab
        self.device = device

    def generate(self, start_token: list, seq_len=10, temperature=1, print_out=True):
        with torch.no_grad():
            self.model.eval()
            tokens = start_token.copy()
            for _ in range(seq_len):
                seq = [self.vocab(j) for j in tokens]
                seq = nn.utils.rnn.pad_sequence(
                    [
                        torch.tensor(seq).clone().detach().to(self.device),
                        torch.zeros(self.model.model_args.seq_max_len),],batch_first=True,)[0]  # 加一项保证padding长度为seq_max_len
                out = self.model(seq.long().unsqueeze(0).to(device))
                out /= temperature
                
                probabilities = F.softmax(out, dim=-1)
                next_token = self.vocab.itos[probabilities.squeeze(0)[len(tokens) - 1].multinomial(num_samples=1)]
                if print_out:
                    print(next_token, end=" ")
                tokens.append(next_token)

        return tokens

In [7]:
MODEL_SAVE_DIR = r"/kaggle/working/"

DATA_DIR = r"/kaggle/input/skypile2022-40-zh-middle-0011/data.txt"

EPOCHS = 1
BATCH = 128

VOCAB_SIZE = 10000
DATASET_DOWNSAMPLE = int(1 / 0.1)
LEARNING_RATE = 8e-4
USE_AMP = False

MODEL = TestModel
args = TestModelArgs(
    d_model=128,
    d_inner=512,
    n_layers=2,
    vocab_size=None,
    seq_max_len=None,
    d_conv=4,
    conv_bias=True,
    bias=False,
    dropout=0.01,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
dataset = TextDatasetV3(DATA_DIR, vocab_size=VOCAB_SIZE, downsample=DATASET_DOWNSAMPLE)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH, shuffle=True, pin_memory=False
)
print(f"词数: {dataset.word_nums}")
dataset_len = len(dataset)
print(f"数据集数量：{dataset_len}")
print(dataset.index2word[:30])

加载并生成训练数据:

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/skypile2022-40-zh-middle-0011/data.txt'

In [ ]:
args.vocab_size = dataset.word_nums
args.seq_max_len = dataset.seq_max_len
model = MODEL(args)

model.to(device)
if torch.cuda.device_count() > 1:
    print("训练显卡数: ", torch.cuda.device_count(), "GPUs!")
    # 就这一行
    model = nn.DataParallel(model)

# print([(i, item) for i, item in enumerate(dataset.index2word)])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, amsgrad=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=EPOCHS, T_mult=2, eta_min=5e-5, last_epoch=-1)
loss_log = []
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("~~~训练咯~~~")

In [ ]:
# 训练模型
for epoch in range(EPOCHS):
    loss_log = []
    loss_sum = 0
    bar = tqdm(dataloader, unit="batch")
    torch.cuda.empty_cache()
    for i, (inputs, targets) in enumerate(dataloader):
        # print(f'Step: {i}')
        model.train()
        inputs = inputs.to(device)
        targets = targets.to(device)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            output = model(torch.as_tensor(inputs))
            # print(inputs.size(), output.size(), targets.size())
            loss = criterion(output.view(-1, dataset.word_nums), targets.view(-1))

        scaler.scale(loss).backward()
        # scaler.unscale_(optimizer)
        # torch.nn.utils.clip_grad_norm_(
        #     parameters=model.parameters(), max_norm=1, norm_type=2
        # )

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        loss_log.append(loss.item())
        loss_sum += loss.item()

        bar.update(1)
        bar.postfix = f"Loss: {round(loss.item(), 5)}"

    scheduler.step()
    bar.close()

    with torch.no_grad():
        print(scheduler.get_last_lr())
        print(
            f"Epoch: {epoch+1}/{EPOCHS}, Loss_sum: {loss_sum}, Loss_avg: {loss_sum/len(dataloader)}"
        )
        # start = [
        #     random.choice(dataset.index2word[2:20])
        # ]  # 取词表中词频最高的20词(不含OOV PAD)
        start = ['我']
        generator = TextGenerator(model, dataset.vocab, device)
        ans = generator.generate(start_token=start, seq_len=25, print_out=False)
        ans = ans[len(start) :]  # 截掉start_token
        print(f"Step: {i} (input){start}->", end="")
        for i in ans:
            print(i, end=" ")
        print('\n\n')
        # 可视化loss曲线
        plt.figure(figsize=(15, 4))
        plt.subplot(1, 2, 1)
        plt.cla()  # 清除当前轴
        plt.plot(loss_log)
        plt.yscale("log")  # 将 y 轴设置为对数坐标轴
        plt.xlabel("Iteration")
        plt.ylabel("Loss")

        plt.subplot(1, 2, 2)
        plt.cla()  # 清除当前轴
        plt.plot(loss_log)
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.pause(0.001)  # 暂停一小段时间以便图形更新
        plt.show()
        if epoch % 1 == 0:
            # 保存ckpt
            torch.save(model, f"{''.join(MODEL_SAVE_DIR.split(r'/')[:-1])}ckpt_{1}.pth")
        


# 保存模型
torch.save(model, f"{MODEL_SAVE_DIR}")
torch.jit.trace(model, inputs).save(f"model_viz.pt")


In [ ]:
test_generator = TextGenerator(model, dataset.vocab, device)
MAX_LEN = 100
T=0.8
while True:
    start = input("In>>")
    if start[:2] == 'T=':
        T = float(start[2:])
        print(f'T={T}')
    else:
        print(f'T={T}\n' +
            "".join(
                test_generator.generate(
                    start_token=list(start),
                    seq_len=MAX_LEN,
                    temperature=T,
                    print_out=False,
                )
            )
        )